# Comprehensive Concurrency

Concurrency is about dealing with multiple things at once. In Python, there are three main approaches:

1. **Threading**: Multiple threads in the same process. Good for I/O-bound tasks.
2. **Multiprocessing**: Multiple distinct Python processes. Good for CPU-bound tasks.
3. **Asyncio**: A single-threaded, event-loop approach. Highly scalable for I/O-bound code.

Let's dive into how these work, the pitfalls you might encounter, and the best practices for using them.

## 1. Threading, Race Conditions, and Locks

Because threads share the same memory space, they can step on each other's toes if they try to modify the same variable at the same time. This is called a **Race Condition**.

To prevent this, we use a **Lock** (`threading.Lock`). A lock ensures that only one thread can execute a specific block of code at a time.

In [ ]:
import threading
import time

database_value = 0
lock = threading.Lock()

def update_database(thread_name):
    global database_value

    # Acquire the lock before reading/writing shared data
    with lock:
        print(f"{thread_name} has the lock. Reading value...")
        local_copy = database_value

        # Simulate some processing time
        local_copy += 1
        time.sleep(0.1)

        database_value = local_copy
        print(f"{thread_name} updated value to {database_value}. Releasing lock.\n")

threads = []
for i in range(3):
    t = threading.Thread(target=update_database, args=(f"Thread-{i+1}",))
    threads.append(t)
    t.start()

for t in threads:
    t.join()

print(f"Final database value: {database_value} (Should be 3)")

## 2. Thread Pools and `as_completed`

Instead of managing threads manually, the `concurrent.futures` module provides Executors. 

Often, tasks take different amounts of time to complete. If we use `map()`, we have to wait for all of them to finish in order. If we use `as_completed()`, we can process results the exact moment each thread finishes, regardless of the order they started.

In [ ]:
import concurrent.futures
import time
import random

def fetch_url(url):
    # Simulate random network latency
    delay = random.uniform(0.5, 2.5)
    time.sleep(delay)
    return f"{url} (took {delay:.2f}s)"

urls = ["google.com", "python.org", "github.com", "stackoverflow.com"]

print("Starting downloads...\n")
start_time = time.time()

with concurrent.futures.ThreadPoolExecutor(max_workers=4) as executor:
    # Submit tasks to the executor (returns Future objects)
    future_to_url = {executor.submit(fetch_url, url): url for url in urls}

    # Yields futures as soon as they complete
    for future in concurrent.futures.as_completed(future_to_url):
        url = future_to_url[future]
        try:
            result = future.result()
            print(f"SUCCESS: {result}")
        except Exception as e:
            print(f"ERROR fetching {url}: {e}")

print(f"\nAll done in {time.time() - start_time:.2f} seconds!")

## 3. Multiprocessing: Bypassing the GIL

Python has a Global Interpreter Lock (GIL) that prevents multiple threads from executing Python bytecode simultaneously. For math-heavy (CPU-bound) code, threading won't speed things up.

We use `ProcessPoolExecutor` to spawn entirely separate Python processes. Each process gets its own memory and its own GIL, unlocking true parallel processing across CPU cores.

In [ ]:
import concurrent.futures
import time
import math

def heavy_computation(number):
    print(f"Processing {number}...")
    result = 0
    # Heavy math loop
    for i in range(15_000_000):
        result += math.sqrt(number)
    return result

numbers = [4, 9, 16, 25]

# Note: In Jupyter, this works out of the box.
# In a standard .py script, this must be wrapped in: if __name__ == '__main__':
start_time = time.time()

with concurrent.futures.ProcessPoolExecutor() as executor:
    # Using map to apply the function to the list of numbers in parallel
    results = executor.map(heavy_computation, numbers)

print("\nComputations finished.")
print(f"ProcessPoolExecutor time: {time.time() - start_time:.2f} seconds")

## 4. Asyncio: Event Loops and Tasks

`asyncio` is fundamentally different. It uses a single thread and an **Event Loop**. When a function hits an `await` statement (like waiting for a database response), it yields control back to the loop so another function can run.

We can create **Tasks** to run things concurrently in the background while we do other work.

In [ ]:
import asyncio
import time

async def brew_coffee():
    print("Start brewing coffee...")
    await asyncio.sleep(3)  # Simulating a non-blocking I/O wait
    print("Coffee is ready!")
    return "Espresso"

async def toast_bagel():
    print("Start toasting bagel...")
    await asyncio.sleep(2)  # Simulating a non-blocking I/O wait
    print("Bagel is ready!")
    return "Everything Bagel"

async def prepare_breakfast():
    start_time = time.time()

    # Create tasks to run concurrently in the background
    coffee_task = asyncio.create_task(brew_coffee())
    bagel_task = asyncio.create_task(toast_bagel())

    print("Doing other morning chores while breakfast cooks...")
    await asyncio.sleep(1)
    print("Finished chores.")

    # Wait for both tasks to complete and get their results
    coffee_result = await coffee_task
    bagel_result = await bagel_task

    print(f"\nBreakfast complete in {time.time() - start_time:.2f}s: {coffee_result} and {bagel_result}")

# In Jupyter, we can await directly.
# In a standard script, use: asyncio.run(prepare_breakfast())
await prepare_breakfast()

## 5. Summary Cheat Sheet

| Approach | When to use | Key Characteristics | Common Modules |
| :--- | :--- | :--- | :--- |
| **Threading** | I/O-bound (web requests, file reading) | Shares memory (needs Locks). Restricted by GIL. | `threading`, `ThreadPoolExecutor` |
| **Multiprocessing** | CPU-bound (math, data processing, ML) | Separate memory space. Bypasses GIL. Heavy OS overhead. | `multiprocessing`, `ProcessPoolExecutor` |
| **Asyncio** | Heavy I/O-bound (high-concurrency web servers, websockets) | Single-threaded. Event loop. Lightweight. Requires `async`/`await` syntax. | `asyncio`, `aiohttp` |